In [2]:
import pandas as pd
import numpy as np
import pickle
import os

# 1. 데이터 로드
path_hosp = '../data/hosp/'

# 파일이 gz로 압축되어 있을 수도 있고 아닐 수도 있으니 처리
try:
    adm = pd.read_csv(os.path.join(path_hosp, 'admissions.csv.gz'))
    diag = pd.read_csv(os.path.join(path_hosp, 'diagnoses_icd.csv.gz'))
except FileNotFoundError:
    adm = pd.read_csv(os.path.join(path_hosp, 'admissions.csv'))
    diag = pd.read_csv(os.path.join(path_hosp, 'diagnoses_icd.csv'))

# 2. 입원 기록에서 필요한 것만 남기기 (환자ID, 입원ID, 입원시간)
adm = adm[['subject_id', 'hadm_id', 'admittime']].copy()
adm['admittime'] = pd.to_datetime(adm['admittime'])

# 3. 진단 기록과 입원 기록 합치기
# hadm_id(입원 건)를 기준으로 합칩니다.
full_data = pd.merge(adm, diag, on=['subject_id', 'hadm_id'], how='inner')

# 4. 정렬: 환자별(subject_id) -> 시간순(admittime) -> 순서(seq_num)
# 이렇게 해야 모델이 "아, 옛날에 이 병이 있어서 나중에 저 병이 생겼구나"를 배웁니다.
full_data = full_data.sort_values(['subject_id', 'admittime', 'seq_num'])

print("전체 데이터 수:", len(full_data))
display(full_data.head())

전체 데이터 수: 4506


,subject_id,hadm_id,admittime,seq_num,icd_code,icd_version
324,10000032,22595853,2180-05-06 22:23:00,1,5723,9
326,10000032,22595853,2180-05-06 22:23:00,2,78959,9
323,10000032,22595853,2180-05-06 22:23:00,3,5715,9
327,10000032,22595853,2180-05-06 22:23:00,4,07070,9
322,10000032,22595853,2180-05-06 22:23:00,5,496,9


In [3]:
# 1. 고유한 진단 코드 추출 (Unique ICD Codes)
unique_codes = full_data['icd_code'].unique()

# 2. 코드 -> 숫자 사전 만들기 (Code to ID Map)
# 예: {'4019': 0, '4280': 1, ...}
code2id = {code: i for i, code in enumerate(unique_codes)}
id2code = {i: code for code, i in code2id.items()}

print(f"총 고유 질병 코드 수: {len(unique_codes)}")

# 3. 데이터프레임에 숫자 코드 컬럼 추가
full_data['code_id'] = full_data['icd_code'].map(code2id)

display(full_data[['subject_id', 'admittime', 'icd_code', 'code_id']].head())

총 고유 질병 코드 수: 1472


,subject_id,admittime,icd_code,code_id
324,10000032,2180-05-06 22:23:00,5723,0
326,10000032,2180-05-06 22:23:00,78959,1
323,10000032,2180-05-06 22:23:00,5715,2
327,10000032,2180-05-06 22:23:00,07070,3
322,10000032,2180-05-06 22:23:00,496,4


In [4]:
# 환자별로 그룹핑해서 리스트로 만들기
# 결과 구조: {환자ID: [[방문1의 질병코드들], [방문2의 질병코드들], ...]}

patient_history = {}

# 환자별로 묶기
for subject_id, group in full_data.groupby('subject_id'):
    # 입원별(hadm_id)로 다시 묶기 (시간순 정렬되어 있음)
    visits = []
    for hadm_id, sub_group in group.groupby('hadm_id', sort=False):
        codes = sub_group['code_id'].tolist()
        visits.append(codes)
    
    patient_history[subject_id] = visits

# 샘플 확인
sample_pid = list(patient_history.keys())[0]
print(f"환자 {sample_pid}의 방문 기록 구조:")
print(patient_history[sample_pid])

환자 10000032의 방문 기록 구조:
[[0, 1, 2, 3, 4, 5, 6, 7], [8, 1, 9, 10, 4, 2, 11, 12], [13, 14, 15, 10, 1, 16, 12, 11, 17, 18, 4, 5, 2], [19, 1, 18, 2, 16, 10, 4, 11, 12, 20]]


In [9]:
import pandas as pd
import numpy as np
import pickle
import os

# 1. 데이터 로드 함수 (gz 처리 포함)
def load_data(path, filename):
    full_path = os.path.join(path, filename)
    if os.path.exists(full_path + '.gz'):
        return pd.read_csv(full_path + '.gz', low_memory=False)
    return pd.read_csv(full_path, low_memory=False)

path_hosp = '../data/hosp/'
print("데이터 로딩 중... (시간이 조금 걸릴 수 있습니다)")

# 기본 테이블 로드
adm = load_data(path_hosp, 'admissions.csv')
diag = load_data(path_hosp, 'diagnoses_icd.csv')
proc = load_data(path_hosp, 'procedures_icd.csv')
presc = load_data(path_hosp, 'prescriptions.csv')

# 2. 필요한 컬럼만 선택 및 정렬
# 입원 기록: 환자, 입원ID, 입원시간
adm = adm[['subject_id', 'hadm_id', 'admittime']].copy()
adm['admittime'] = pd.to_datetime(adm['admittime'])
adm = adm.sort_values(['subject_id', 'admittime'])

# 3. 데이터 매핑 (String -> Integer ID)
# 컴퓨터는 글자를 모르니 숫자로 바꿔주는 '사전(Vocab)'을 만듭니다.

def create_vocab(df, col_name):
    unique_codes = df[col_name].unique()
    code2id = {code: i for i, code in enumerate(unique_codes)}
    return code2id

# (1) 진단 코드 사전 (Conditions)
diag_vocab = create_vocab(diag, 'icd_code')
# (2) 시술 코드 사전 (Procedures)
proc_vocab = create_vocab(proc, 'icd_code')
# (3) 약물 이름 사전 (Drugs) - 약물은 종류가 많아 'drug' 이름 자체를 씁니다.
presc_vocab = create_vocab(presc, 'drug')

print(f"✅ 사전 생성 완료:")
print(f" - 진단(Condition) 종류: {len(diag_vocab)}개")
print(f" - 시술(Procedure) 종류: {len(proc_vocab)}개")
print(f" - 약물(Drug) 종류: {len(presc_vocab)}개")

# 4. 환자별 데이터 구조화 (핵심!)
# 목표 구조: {환자ID: [방문1 데이터, 방문2 데이터, ...]}
# 방문 데이터: {'C': [진단코드들], 'P': [시술코드들], 'D': [약물코드들]}

patient_history = {}

# 속도를 위해 각 테이블을 hadm_id 기준으로 미리 그룹화
diag_grp = diag.groupby('hadm_id')['icd_code'].apply(list)
proc_grp = proc.groupby('hadm_id')['icd_code'].apply(list)
presc_grp = presc.groupby('hadm_id')['drug'].apply(list)

count = 0
for subject_id, group in adm.groupby('subject_id'):
    visits = []
    
    # 환자의 방문(입원) 기록을 시간 순서대로 순회
    for _, row in group.iterrows():
        hadm_id = row['hadm_id']
        
        # 각 테이블에서 해당 입원ID(hadm_id)에 맞는 코드 찾아오기
        # (1) 진단 코드 리스트 -> ID 변환
        c_list = [diag_vocab[c] for c in diag_grp.get(hadm_id, []) if c in diag_vocab]
        
        # (2) 시술 코드 리스트 -> ID 변환
        p_list = [proc_vocab[p] for p in proc_grp.get(hadm_id, []) if p in proc_vocab]
        
        # (3) 약물 코드 리스트 -> ID 변환
        d_list = [presc_vocab[d] for d in presc_grp.get(hadm_id, []) if d in presc_vocab]
        
        # 방문 데이터 저장
        visit_data = {
            'hadm_id': hadm_id,
            'admittime': row['admittime'],
            'conditions': c_list,
            'procedures': p_list,
            'drugs': d_list
        }
        visits.append(visit_data)
    
    patient_history[subject_id] = visits
    count += 1

print(f"\n🎉 총 {count}명의 환자 데이터 전처리 완료!")

# 5. 샘플 확인 (첫 번째 환자의 첫 번째 방문)
sample_pid = list(patient_history.keys())[0]
sample_visit = patient_history[sample_pid][0]

print(f"\n[샘플 환자 {sample_pid}의 첫 번째 방문 기록]")
print(f"- 진단(C) 개수: {len(sample_visit['conditions'])}")
print(f"- 시술(P) 개수: {len(sample_visit['procedures'])}")
print(f"- 약물(D) 개수: {len(sample_visit['drugs'])}")

데이터 로딩 중... (시간이 조금 걸릴 수 있습니다)
✅ 사전 생성 완료:
 - 진단(Condition) 종류: 1472개
 - 시술(Procedure) 종류: 352개
 - 약물(Drug) 종류: 631개

🎉 총 100명의 환자 데이터 전처리 완료!

[샘플 환자 10000032의 첫 번째 방문 기록]
- 진단(C) 개수: 8
- 시술(P) 개수: 1
- 약물(D) 개수: 14


In [10]:
# 6. 전처리된 데이터와 사전을 파일로 저장
save_path = '../data/processed_data.pkl'

data_to_save = {
    'patient_history': patient_history,
    'vocabs': {
        'diag': diag_vocab,
        'proc': proc_vocab,
        'drug': presc_vocab
    }
}

with open(save_path, 'wb') as f:
    pickle.dump(data_to_save, f)

print(f"💾 데이터가 '{save_path}'에 저장되었습니다.")

💾 데이터가 '../data/processed_data.pkl'에 저장되었습니다.
